## Tutorial of Pipeline Run

#### Import Libraries

In [1]:
import os 
import sys
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from setfit import SetFitModel
from rich import print
import pandas as pd
import torch
from tqdm import tqdm

import gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from torch import bfloat16
from rich import print
import pickle 

from utils.dspy_qwen2 import (JobPostingModule, 
                              QUESTIONANSWER, 
                              validate_ans)


from utils.prepo import (process_texts,
                         count_words,
                         create_dataframe,
                         filter_and_merge_data)

from langchain_experimental.text_splitter import SemanticChunker
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.embeddings import HuggingFaceBgeEmbeddings

/Users/justinvhuang/miniconda3/envs/dspy/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


#### Load in Modules from DSPy 
    * uncompiled_fs is faster
    * uncompiled_module is more precise

In [2]:
uncompiled_fs=QUESTIONANSWER('''
                            "position_title" : What is the title of this position?
                            "location" : Where is this position located, including city, state and zip code?
                            "work_arrange" : What is the work arrangement for this position, remote, hybrid, or on-site?
                            "experience" : what are years of experience required for this position?
                            "employment_type" : What is the employment type, full time, part time, or internship?
                            "pay" : What is the pay for this position?
                            "degree" : What is required degree?
                            "certifications" : What certifications or qualifications are required?
                            "required_skills" : What are required skills?
                              ''')
uncompiled_module = JobPostingModule()

#### Load in LLM for DSPy Configuration, Where LLM can be selected among Ollama-Qwen2, Ollama-Gemmma, GPT3.5, and Gemini1.5, 

In [3]:
# Ollama 

llm = dspy.OllamaLocal(model='qwen2:latest', max_tokens = 1000, temperature=0.0)
dspy.settings.configure(lm=llm)

# HuggingFace 4-Bit Pre-Quant 

# access_token = ""
# model_name = "google/gemma-7b"
# llm = dspy.HFModel(model=model_name, hf_device_map='auto', token=access_token, model_kwargs= {'temperature': 0.0, 'do_sample': False})
# llm.model=None
# gc.collect()

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,  # 4-bit quantization
#     bnb_4bit_quant_type='nf4',  # Normalized float 4
#     bnb_4bit_use_double_quant=True,  # Second quantization after the first
#     bnb_4bit_compute_dtype=bfloat16  # Computation type
# )
# llm.model=AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
# dspy.settings.configure(lm=llm)

# Hugging Face Post-Quant 4 Bit from Unsloth

# llm = dspy.HFModel(model="unsloth/Qwen2-7B-bnb-4bit", hf_device_map='auto', model_kwargs= {'temperature':0.0,'do_sample': False})
# dspy.settings.configure(lm = llm)

# Connect OpenAI gpt3.5 turbo to DSPy

# api_key=os.getenv('openai_key')
# llm = dspy.OpenAI(model='gpt-3.5-turbo', api_key=api_key)
# dspy.settings.configure(lm=llm)

# Connect to Gemini 1.5 

# gemini_key=os.getenv('gemini_key')
# gemini = dspy.Google("gemini-1.5-flash-latest", api_key=gemini_key)
# dspy.settings.configure(lm=gemini, max_tokens=1024)

#### Load in Example Data From LightCast

In [4]:
df = pd.read_csv("~/data/sample_unclassified_postings.csv", index_col=0)

In [5]:
df[['id', 'body']].head()

,id,body
41066,793b547eed99250aa1b1c3ee30102550f6772919,Vocational Coordinator Kennedy Day School Fran...
17423,05199a06702bfe05fd762f0d4e3b2c464d22e690,"Plastic Surgery Scheduler San Francisco, CA $2..."
112080,fbbd39619c18ea664c3c0a5410eeca9e916b4947,Avionics Expert (SME)\nAvionics Expert (SME)\n...
15434,47a2072651eaf28604ca7e947e78ca6fa7798d61,"Used Car Techs, $21 - $30/hour, Tool Relocatio..."
58984,97e440ec22c524ac8e04ba3cb0216bf7928ce347,"Cyberspace Operations Officer\nFreeport, Illin..."


#### Covnert To Dictionary to Save ID and Body

In [6]:
text_dict = {key:value for key, value in zip(df['id'], df['body'])}

#### Choose Between Semantic Chunking and Recursive Character Text Splitting - Semantic - Less Than 1500 words and less than 8000 Tokens otherwise Recursive Character Text Spltting or else will OOM on 16 GB VRAM

In [7]:
#Semantic Chunking Model
model_name = "nomic-ai/nomic-embed-text-v1.5"
#Set to device to cpu or cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_kwargs = {'device': device, 'trust_remote_code': True}
#normalize to embed faster 
encode_kwargs = {'normalize_embeddings': True}

#Use LangChain to do text embeddings
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

/Users/justinvhuang/miniconda3/envs/dspy/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
#Semantic Text Spliter
semantic_text_splitter = SemanticChunker(embeddings=embeddings,
                                 breakpoint_threshold_type="percentile",
                                 breakpoint_threshold_amount=5,
                                 )

#Recursive Text Splitter
recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

#### Split Text and Examine will create a Dictionary Key Value pair with the Key: ID and Value: Job Posting Split into strings within a list
    * please review text-chunking.ipynb there it shows splitting between long job postings and short here we only show semantic chunking for simplicity

In [9]:
split_text_dict = process_texts(text_dict, semantic_text_splitter)

100%|██████████| 20/20 [01:30<00:00,  4.54s/it]


In [10]:
print(len(split_text_dict['793b547eed99250aa1b1c3ee30102550f6772919']))
split_text_dict['793b547eed99250aa1b1c3ee30102550f6772919']

31

["Vocational Coordinator Kennedy Day School Franciscan Children's - Brighton, MA Apply Now Job Details Full-time Estimated: $41,000 - $54,000 a year 9 hours ago Full Job Description 1). Develop life skills curriculum that implements all aspects of life skills and post-school endeavors.",
 '2).',
 'Collaborates with school therapists and specialists to develop job training sites within the school and community.',
 '3).',
 'Consult and collaborate with classroom teachers to integrate life skills and vocational activities in the classroom.',
 '4).',
 'Write and report on vocational and life skills oriented IEP goals.',
 '5).',
 'Attends team meetings, individual education plan meetings (IEP), faculty meetings, coordinators meetings and any other requested meetings.',
 '6).',
 'Prepares and utilizes assessment methodologies to adapt for individual students as appropriate.',
 '7).',
 'Collaborate with families to help implement life skills strategies/routines at home.',
 '8).',
 'Actively p

#### Load in SetFit Model that was trained on Labeled data

In [11]:
model_name = "setfit_model_v5"
model = SetFitModel.from_pretrained(f"~/ISYE-CSE-MGT-6748-Group-1/models/{model_name}", trust_remote_code = True)

In [12]:
data1 = create_dataframe(split_text_dict)

#### Perform Text Denoising within list of strings

In [13]:
labels = [model.predict([sent]) for sent in tqdm(data1['text'])]

100%|██████████| 468/468 [00:36<00:00, 12.72it/s]


In [14]:
filtered_original_data, new_data, data1_filtered = filter_and_merge_data(data1, labels, df, count_words)

#### Examine Results

In [15]:
new_data[['id','body', 'word_count_original', 'signals_text','word_count_signals', 
              'no_signals_text', 'word_count_no_signal']].head()

,id,body,word_count_original,signals_text,word_count_signals,no_signals_text,word_count_no_signal
0,0138edfb8a54382c8c42fb889632c2e503cb59d1,Assistant Manager - Hotel Parking Operations 6...,924,Assistant Manager - Hotel Parking Operations 6...,446,Manage job preferences anytime in your . You a...,478.0
1,0213598d286572c89f80c7e15722688ed66fd0b3,"Inbound Logistics Coordinator Columbus, OH $20...",513,"Inbound Logistics Coordinator Columbus, OH $20...",421,Yes No Job details Here's how the job details ...,92.0
2,05199a06702bfe05fd762f0d4e3b2c464d22e690,"Plastic Surgery Scheduler San Francisco, CA $2...",384,"Plastic Surgery Scheduler San Francisco, CA $2...",306,Yes No Job details Heres how the job details a...,78.0
3,135d3d0c1f9654d02703bf2dd9dfe08c579ac504,AVP - Underwriting Company: Farm Credit Founda...,892,AVP - Underwriting Company: Farm Credit Founda...,661,We are a growing organization embracing collab...,231.0
4,2efd8e2c7eb393129bff7bd48c942fbd6ac528d0,Pharmacy Technician I Burrell Behavioral Healt...,595,Pharmacy Technician I Burrell Behavioral Healt...,480,"Springfield offers a high quality of life, wit...",115.0


#### More Precise Using ChainOfThought with Hint and Pydantic to Split up Information Search - Takes about 23 Seconds to run

In [16]:
with dspy.context(lm = llm):
    pred = uncompiled_module(job_posting =new_data['signals_text'][0])
    print(pred.info_extracted)

{
    'position_title': 'Assistant Account Manager',
    'location': 'Denver, CO',
    'work_arrangement': 'On-site',
    'experience': '5+ years',
    'employment_type': 'Full-time',
    'pay': '$68K - $75K/year',
    'degree': "Bachelor's",
    'certifications': 'Previous parking industry exp., hospitality focus, supervisory exp., front-line team 
oversight, scheduling & payroll skills, Bachelors in business.',
    'required_skills': 'Management, parking ops, customer service, hospitality exp, supervisory skills, scheduling,
payroll, comm., problem-solving, training, admin. duties, safety, finance, policy compliance.'
}

#### Faster but less precise might not keep dictionary format - Takes about 9-10 seconds to run

In [17]:
with dspy.context(lm = llm):
    pred = uncompiled_fs(context =new_data['signals_text'][0])
    print(pred.answer)

{
  "position_title": "Assistant Account Manager",
  "location": "650 15th street, Denver, CO 80202",
  "work_arrangement": "on-site",
  "experience": "previous parking industry experience with a hospitality focus is ideal",
  "employment_type": "full-time",
  "pay": "$68,000 - $75,000 a year",
  "degree": "bachelors degree with a concentration in business is helpful but not required",
  "certifications": "not specified",
  "required_skills": "management experience in the parking, customer service, and/or hospitality industries"
}

#### Store Dictionary to be used for other downstream tasks

In [18]:
with dspy.context(lm = llm):
    pred_dict_precise = {key: uncompiled_module(job_posting = value).info_extracted for key, value in zip(new_data['id'],new_data['signals_text'])}

In [22]:
pred_dict_precise

{'0138edfb8a54382c8c42fb889632c2e503cb59d1': {'position_title': 'Assistant Account Manager',
  'location': 'Denver, CO',
  'work_arrangement': 'On-site',
  'experience': '5+ years',
  'employment_type': 'Full-time',
  'pay': '$68K - $75K/year',
  'degree': "Bachelor's",
  'certifications': 'Previous parking industry exp., hospitality focus, supervisory exp., front-line team oversight, scheduling & payroll skills, Bachelors in business.',
  'required_skills': 'Management, parking ops, customer service, hospitality exp, supervisory skills, scheduling, payroll, comm., problem-solving, training, admin. duties, safety, finance, policy compliance.'},
 '0213598d286572c89f80c7e15722688ed66fd0b3': {'position_title': 'Inbound Logistics Coordinator',
  'location': 'Columbus, OH',
  'work_arrangement': 'On-site',
  'experience': '2-4 years',
  'employment_type': 'Full-time',
  'pay': '$20 - $30 per hour',
  'degree': 'High School Diploma or GED',
  'certifications': "High School Diploma/GED, 1-3 y

In [19]:
with open('~/ISYE-CSE-MGT-6748-Group-1/data/pred_dict_precise.pickle', 'wb') as handle:
    pickle.dump(pred_dict_precise, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [20]:
with dspy.context(lm = llm):
    pred_dict_fast = {key: uncompiled_fs(context = value).answer for key, value in zip(new_data['id'],new_data['signals_text'])}

In [23]:
pred_dict_fast

{'0138edfb8a54382c8c42fb889632c2e503cb59d1': '{\n  "position_title": "Assistant Account Manager",\n  "location": "650 15th street, Denver, CO 80202",\n  "work_arrangement": "on-site",\n  "experience": "previous parking industry experience with a hospitality focus is ideal",\n  "employment_type": "full-time",\n  "pay": "$68,000 - $75,000 a year",\n  "degree": "bachelors degree with a concentration in business is helpful but not required",\n  "certifications": "not specified",\n  "required_skills": "management experience in the parking, customer service, and/or hospitality industries"\n}',
 '0213598d286572c89f80c7e15722688ed66fd0b3': '{\n    "position_title": "Inbound Logistics Coordinator",\n    "location": "Columbus, OH",\n    "work_arrange": "On-site",\n    "experience": "1-3 years\' experience",\n    "employment_type": "Full-time",\n    "pay": "$20 - $30 an hour",\n    "degree": "High school diploma or equivalent; preferred: 2-4-year degree in supply chain, logistics, transportation 

In [21]:
with open('~/ISYE-CSE-MGT-6748-Group-1/data/pred_dict_fast.pickle', 'wb') as handle:
    pickle.dump(pred_dict_fast, handle, protocol=pickle.HIGHEST_PROTOCOL)